# 3. 工具的定义方式2：使用 @tool 装饰器（推荐）

使用 @tool 装饰器修饰，可以自动将普通 Python 函数转化为智能体可调用的工具。

此方式 最直接 ，代码量极少，非常适合快速验证想法或创建参数简单的工具。

本节分两部分：

- **3.1 原理**：加上 `@tool` 之后，函数变成了什么；调用工具时又发生了什么
- **3.2 常用参数**：`description`、`parse_docstring`、`name_or_callable`、`args_schema`、`return_direct`、`response_format`

## 3.1 装饰器让函数变成工具的原理

### 3.1.1 先复习：装饰器就是"用返回值替换原函数"

装饰器的写法：

```python
@tool
def get_weather(city: str): ...
```

完全等价于：

```python
def get_weather(city: str): ...
get_weather = tool(get_weather)   # 把函数交给 tool()，再用它的返回值覆盖 get_weather 这个名字
```

关键在第二行：**`get_weather` 这个名字最后指向的是 `tool()` 的返回值，而不是原来的函数。** 装饰器返回什么，被装饰的名字就变成什么。

写一个"捣乱"的装饰器就能看清这一点：

In [1]:
def replace_with_text(func):
    print(f"装饰器收到了函数：{func.__name__}")
    return "我已经不是函数了"   # 这个返回值会替换掉原函数

@replace_with_text
def hello():
    return "hello"

print(hello)
print(type(hello))

装饰器收到了函数：hello
我已经不是函数了
<class 'str'>


两个现象：

1. `hello` 变成了字符串，原来的函数被替换掉了
2. 第一行输出出现在**定义函数的时候**，此时还没有调用 `hello`。装饰器在定义函数的那一刻就执行了，所以 `@tool` 生成工具说明也发生在定义时，不是调用时

**带参数的装饰器**多了一层调用。`@tool(parse_docstring=True)` 等价于：

```python
get_weather = tool(parse_docstring=True)(get_weather)
#             └──── 第 1 次调用 ────┘└─ 第 2 次调用 ─┘
#             先用参数调用 tool()，得到一个"装饰器"；再用这个装饰器去装饰函数
```

所以 `tool()` 要同时支持两种用法：直接收函数（`@tool`），或者先收参数、返回一个装饰器（`@tool(...)`）。它靠**第一个参数的类型**来区分，这也是第一个参数叫 `name_or_callable`（名字或可调用对象）的原因：

| 写法 | `name_or_callable` 收到的是 | `tool()` 的处理 |
|---|---|---|
| `@tool` | 函数本身 | 用函数名当工具名，**直接创建工具** |
| `@tool("getWeather")` | 字符串 | 记下工具名，**返回一个装饰器**，等着接收函数 |
| `@tool(parse_docstring=True)` | `None`（没传） | **返回一个装饰器**，收到函数后用函数名当工具名 |

对应的源码（`langchain_core/tools/convert.py`，有删减）：

```python
if name_or_callable is not None:
    if callable(name_or_callable) and hasattr(name_or_callable, "__name__"):
        # @tool：收到的是函数，直接创建工具
        return _create_tool_factory(name_or_callable.__name__)(name_or_callable)
    if isinstance(name_or_callable, str):
        # @tool("search")：收到的是名字，返回装饰器
        return _create_tool_factory(name_or_callable)

# @tool(parse_docstring=True)：第一个参数什么都没收到，返回装饰器
def _partial(func):
    tool_factory = _create_tool_factory(func.__name__)
    return tool_factory(func)
return _partial
```

三种写法最后都走到 `_create_tool_factory(工具名)(函数)`，由它真正创建工具。

### 3.1.2 装饰之后，get_weather 变成了什么

`_create_tool_factory` 创建的是一个 `StructuredTool` 对象，它是 `BaseTool` 的子类。用代码看看：

In [2]:
from langchain.tools import tool
from langchain_core.tools import BaseTool

@tool
def get_weather(city: str, include_forecast: bool = False) -> str:
    """获取指定城市的天气"""
    result = f"{city}天气晴朗"
    if include_forecast:
        result += "，未来五天都是晴天"
    return result

print("类型:", type(get_weather).__name__, "| 是 BaseTool 吗:", isinstance(get_weather, BaseTool))
print("name:", get_weather.name)
print("description:", get_weather.description)
print("args_schema:", get_weather.args_schema)
print("args:", get_weather.args)
print("func:", get_weather.func)

类型: StructuredTool | 是 BaseTool 吗: True
name: get_weather
description: 获取指定城市的天气
args_schema: <class 'langchain_core.utils.pydantic.get_weather'>
args: {'city': {'title': 'City', 'type': 'string'}, 'include_forecast': {'default': False, 'title': 'Include Forecast', 'type': 'boolean'}}
func: <function get_weather at 0x107ee87c0>


装饰后的 `get_weather` 是一个**工具对象**，原来的函数被收进了它的 `func` 属性里：

| 属性 | 来源 | 作用 |
|---|---|---|
| `name` | 函数名（或 `@tool("名字")`） | 工具名，发给模型 |
| `description` | docstring（或 `description` 参数） | 工具描述，发给模型 |
| `args_schema` | 根据函数签名**生成的 Pydantic 模型**（类名就是工具名） | 发给模型的参数说明，也用来校验参数 |
| `args` | `args_schema` 的简要视图 | 方便查看参数 |
| `func` | 原来的函数 | 真正干活的代码 |
| `return_direct`、`response_format` 等 | `@tool` 的参数 | 控制执行结果怎么返回（见 3.2） |

对照上一节：普通函数要等到 `bind_tools` 时才"现场"生成说明书；`@tool` 在**定义函数的那一刻**就生成好了，存在工具对象上。`bind_tools` 遇到工具对象时，走 `convert_to_openai_function` 的 `BaseTool` 分支，直接读取这些现成的属性。

### 3.1.3 tool() 内部做了什么

`_create_tool_factory` 把函数交给 `StructuredTool.from_function`，核心代码如下（`langchain_core/tools/structured.py`，有删减）：

```python
def from_function(func, name, description, args_schema, parse_docstring, ...):
    # ① 工具名：没传就用函数名
    name = name or func.__name__

    # ② 参数模型：没传 args_schema，就根据函数签名生成（和普通函数用的是同一个函数）
    if args_schema is None:
        args_schema = create_schema_from_function(name, func, parse_docstring=parse_docstring, ...)

    # ③ 工具描述：优先用 description 参数，其次用 docstring
    description_ = description
    if description is None and not parse_docstring:
        description_ = func.__doc__            # 整段 docstring
    if description_ is None and args_schema:
        description_ = args_schema.__doc__     # 解析出的描述，或 args_schema 类自己的 docstring
    if description_ is None:
        raise ValueError("Function must have a docstring if description not provided.")

    # ④ 打包成工具对象
    return cls(name=name, func=func, args_schema=args_schema, description=description_,
               return_direct=return_direct, response_format=response_format, ...)
```

这四步正好对应 3.2 要讲的参数：

| 步骤 | 相关参数 |
|---|---|
| ① 工具名 | `name_or_callable` |
| ② 参数模型 | `args_schema`、`parse_docstring` |
| ③ 工具描述 | `description`、`parse_docstring` |
| ④ 打包 | `return_direct`、`response_format` 原样存到工具对象上，执行时才用到 |

第 ③ 步最后的 `raise`，就是 3.2.1 第一个例子报错的来源：没有 docstring、也没传 `description`，工具就没有描述，LangChain 直接拒绝创建。

### 3.1.4 调用工具时发生了什么

工具对象不能再用 `()` 直接调用，要用 `.invoke()`。和直接调用函数相比，`invoke` 多做了三件事：

1. **校验参数**：先用 `args_schema`（Pydantic 模型）检查、转换参数，不合格就抛出 `ValidationError`，根本不会执行函数
2. **调用原函数**：校验通过后，才调用 `func`
3. **包装结果**：如果传入的是模型返回的 tool_call，就把结果包装成 `ToolMessage`，并自动带上 `tool_call_id`

另外，`BaseTool` 是 `Runnable`，每次 `invoke` 都会触发回调。开启 LangSmith 时，每次调用都会记录为一次工具运行。

In [3]:
from pydantic import ValidationError

# 1. 普通调用：传入参数字典
print(get_weather.invoke({"city": "北京"}))

# 2. 校验并转换：字符串 "true" 按参数类型被转换成了布尔值 True
print(get_weather.invoke({"city": "北京", "include_forecast": "true"}))

# 3. 校验失败：缺少必填参数 city，函数不会被执行
try:
    get_weather.invoke({})
except ValidationError as e:
    print("ValidationError ->", " ".join(line.strip() for line in str(e).splitlines()[1:3]))

# 4. 工具对象不能直接调用
try:
    get_weather("北京")
except TypeError as e:
    print("TypeError ->", e)

# 原函数还在 func 属性里，可以直接调用（不经过校验）
print(get_weather.func("北京"))

北京天气晴朗
北京天气晴朗，未来五天都是晴天
ValidationError -> city Field required [type=missing, input_value={}, input_type=dict]
TypeError -> 'StructuredTool' object is not callable
北京天气晴朗


最实用的是**直接传入 tool_call**。模型返回的 `tool_calls` 里，每一项都是 `{"name", "args", "id", "type": "tool_call"}` 结构。把它直接交给 `invoke`，得到的就是可以回传给模型的 `ToolMessage`：

In [4]:
# 模拟模型返回的一个 tool_call
tool_call = {"name": "get_weather", "args": {"city": "北京"}, "id": "call_123", "type": "tool_call"}

tool_message = get_weather.invoke(tool_call)
print(repr(tool_message))

ToolMessage(content='北京天气晴朗', name='get_weather', tool_call_id='call_123')


对比 `01-Tools_usage.ipynb` 1.4 节里用普通函数手动拼 `ToolMessage` 的写法：

```python
# 普通函数：自己取参数、自己拼 ToolMessage
ToolMessage(content=get_weather(**tool_call["args"]), tool_call_id=tool_call["id"], name="get_weather")

# @tool：一行搞定
get_weather.invoke(tool_call)
```

### 3.1.5 小结：从函数到工具

```text
def get_weather(city: str): """获取指定城市的天气"""
        │  @tool 等价于 get_weather = tool(get_weather)，在定义函数时就执行
        ▼
tool()：看第一个参数判断用法 → _create_tool_factory → StructuredTool.from_function
        │  ① name ← 函数名          ② args_schema ← 根据签名生成的 Pydantic 模型
        │  ③ description ← docstring    ④ 打包成 StructuredTool
        ▼
get_weather 变成 StructuredTool 对象（原函数在 .func 里）
        ├─ bind_tools 时：读取 name / description / args_schema，翻译成 JSON 发给模型
        └─ 执行时：invoke → 用 args_schema 校验参数 → 调用 func → 返回结果（传入 tool_call 时返回 ToolMessage）
```

## 3.2 @tool 的常用参数

完整签名：

```python
tool(
    name_or_callable: str | Callable | None = None,
    runnable: Runnable | None = None,
    *args,
    description: str | None = None,
    return_direct: bool = False,
    args_schema: ArgsSchema | None = None,
    infer_schema: bool = True,
    response_format: Literal["content", "content_and_artifact"] = "content",
    parse_docstring: bool = False,
    error_on_invalid_docstring: bool = True,
    extras: dict[str, Any] | None = None,
)
```

| 参数 | 作用 | 默认值 | 常用程度 |
|---|---|---|---|
| `description` | 自定义工具描述，优先级高于 docstring | `None` | 常用，见 3.2.1 |
| `parse_docstring` | 把 Google 风格的 `Args:` 解析成参数说明 | `False` | 常用，见 3.2.2 |
| `error_on_invalid_docstring` | `parse_docstring=True` 时，docstring 不合规是否报错 | `True` | 常用，见 3.2.2 |
| `name_or_callable` | 工具名（不带括号使用时，收到的是函数本身） | `None` | 常用，见 3.2.3 |
| `args_schema` | 用 Pydantic 模型定义参数 | `None` | 常用，见 3.2.4 |
| `return_direct` | 在 Agent 中，工具执行完直接结束，把结果作为最终答案 | `False` | 常用，见 3.2.5 |
| `response_format` | 让工具返回"给模型的内容 + 附件" | `"content"` | 常用，见 3.2.6 |
| `runnable` | 把一个 Runnable（比如一条链）转换成工具 | `None` | 少用 |
| `infer_schema` | 是否根据函数签名推断参数。设为 `False` 时，退化为只接收一个字符串的简单工具 | `True` | 少用 |
| `extras` | 附加到工具定义上的厂商专用字段 | `None` | 少用 |

下面逐个介绍常用参数。

### 3.2.1 description：自定义工具描述

@tool 装饰器有一个可选的 description 参数，用于自定义工具的描述。

如果未提供 description 参数，默认会使用函数的文档字符串作为描述。

情况一：既没有 docstring，也没有传 description，会报错：

In [5]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool

try:
    @tool
    def get_weather(city: str):
        return f"{city}天气晴朗"
    print(convert_to_openai_tool(get_weather))
except ValueError as e:
    print("ValueError:", e)

ValueError: Function must have a docstring if description not provided.


报错来自 3.1.3 第 ③ 步：工具必须有描述，因为模型要靠描述判断什么时候调用它。注意报错发生在**定义函数时**（`@tool` 那一行），还没轮到 `convert_to_openai_tool`。

情况二：补充文档字符串后：

In [6]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
@tool
def get_weather(city: str):
    """天气查询工具"""
    return f"{city}天气晴朗"

print(convert_to_openai_tool(get_weather))

{'type': 'function', 'function': {'name': 'get_weather', 'description': '天气查询工具', 'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}}}


情况三：添加工具描述后：

@tool 的参数 description 可以更改工具描述，优先级高于 docstring 的函数说明

In [7]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
from rich import print as rprint
@tool(description="根据城市名称查询当日天气的工具")
def get_weather(city: str):
    """天气查询工具"""
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '根据城市名称查询当日天气的工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

工具描述的来源，按优先级排列：

| 情况 | description 取值 |
|---|---|
| 传了 `description` | 以它为准。即使同时设置了 `parse_docstring=True` 也一样，不过 `Args:` 仍会被解析成参数说明 |
| 没传，`parse_docstring=False`（默认） | 整段 docstring，包括 `Args:` 部分（见 3.2.2） |
| 没传，`parse_docstring=True` | docstring 中 `Args:` 之前的部分 |
| 没传，也没有 docstring | 报错。例外：传了 `args_schema` 时，会改用 `args_schema` 类的 docstring（见 3.2.4） |

### 3.2.2 parse_docstring：解析 docstring

当我们没有向 @tool 传递 description 参数时，默认情况下， tool 会将 docstring 整体视为description ，如下：

In [8]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
@tool
def get_weather(city: str, units: str = "celsius", include_forecast: bool =
False) -> str:
    """
    获取当日天气，可选择是否同时查询未来五日天气预报
    
    Args:
        city: 城市
        units: 气温单位，可选：celsius-摄氏度，fahrenheit-华氏度
        include_forecast: 是否包含未来五日的天气预报
    """
    temp = 22 if units == "celsius" else 72
    result = f'{city}当天气温: {temp} {"摄氏度" if units == "celsius" else "华氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result
rprint(convert_to_openai_tool(get_weather))  


{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气，可选择是否同时查询未来五日天气预报\n\nArgs:\n    city: 城市\n    units: 
气温单位，可选：celsius-摄氏度，fahrenheit-华氏度\n    include_forecast: 是否包含未来五日的天气预报',
        'parameters': {
            'properties': {
                'city': {'type': 'string'},
                'units': {'default': 'celsius', 'type': 'string'},
                'include_forecast': {'default': False, 'type': 'boolean'}
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

可以看到，`Args:` 部分被原样塞进了 description，参数本身却没有说明。这一点和上一节的普通函数不同：普通函数默认会解析 `Args:`，而 `@tool` 默认不解析。

通过将 parse_docstring 设置为True，docstring会被解析，填充到相应的字段描述中。

In [9]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
@tool(parse_docstring=True)
def get_weather(city: str, units: str = "celsius", include_forecast: bool =
False) -> str:
    """
    获取当日天气，可选择是否同时查询未来五日天气预报
    
    Args:
        city: 城市
        units: 气温单位，可选：celsius-摄氏度，fahrenheit-华氏度
        include_forecast: 是否包含未来五日的天气预报
    """
    temp = 22 if units == "celsius" else 72
    result = f'{city}当天气温: {temp} {"摄氏度" if units == "celsius" else "华氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result
rprint(convert_to_openai_tool(get_weather))  


{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气，可选择是否同时查询未来五日天气预报',
        'parameters': {
            'properties': {
                'city': {'description': '城市', 'type': 'string'},
                'units': {
                    'default': 'celsius',
                    'description': '气温单位，可选：celsius-摄氏度，fahrenheit-华氏度',
                    'type': 'string'
                },
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五日的天气预报',
                    'type': 'boolean'
                }
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

**什么样的 docstring 才能被解析？** LangChain 的解析规则很简单（源码见 `_parse_google_docstring`）：

1. 先按**空行**把 docstring 切成若干"块"
2. 第一块是工具描述；后面必须有一块以 `Args:` 开头
3. `Args:` 块里，每行写 `参数名: 说明`；缩进更深的行，算作上一个参数说明的续行
4. `Returns:`、`Raises:` 等块会被跳过，不会发给模型

`parse_docstring=True` 时，只要函数有参数，就必须满足第 2 条，否则报错 `Found invalid Google-Style docstring.`。常见的不合规写法：

| 写法 | 问题 |
|---|---|
| 描述和 `Args:` 之间没有空行 | 切出来只有一块，找不到单独的 `Args:` 块 |
| 用中文 `参数:` 代替 `Args:` | 没有以 `Args:` 开头的块 |
| 只写一句描述，没有 `Args:` | 同上。函数有参数时，开了 `parse_docstring` 就必须写 `Args:` |

要注意：不使用 @tool 装饰器时（普通函数），docstring 不合规会被视为普通文本，作为 description；使用 `@tool(parse_docstring=True)` 时，docstring 不合规将会抛出异常。

下面的 docstring 在描述和 `Args:` 之间少了一个空行：

In [10]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

try:
    @tool(parse_docstring=True)
    def get_weather(city: str, units: str = "celsius", include_forecast: bool = False) -> str:
        """
        获取当日天气，可选择是否同时查询未来五日天气预报
        Args:
            city: 城市
            units: 气温单位，可选：celsius-摄氏度，fahrenheit-华氏度
            include_forecast: 是否包含未来五日的天气预报
        """
        temp = 22 if units == "celsius" else 72
        result = f'{city}当天气温: {temp} {"摄氏度" if units == "celsius" else "华氏度"}'
        if include_forecast:
            result += "\n未来五天都是晴天"
        return result
    rprint(convert_to_openai_tool(get_weather))
except ValueError as e:
    print("ValueError:", e)

ValueError: Found invalid Google-Style docstring.


**error_on_invalid_docstring：不合规时是否报错。** 设为 `False` 后，docstring 不合规时不再报错，而是退回"整段 docstring 当描述"的做法：

In [11]:
@tool(parse_docstring=True, error_on_invalid_docstring=False)
def get_weather(city: str, units: str = "celsius") -> str:
    """
    获取当日天气
    Args:
        city: 城市
        units: 气温单位，可选：celsius-摄氏度，fahrenheit-华氏度
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气\nArgs:\n    city: 城市\n    units: 
气温单位，可选：celsius-摄氏度，fahrenheit-华氏度',
        'parameters': {
            'properties': {'city': {'type': 'string'}, 'units': {'default': 'celsius', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

实际上，**上一节普通函数的解析方式，就相当于 `@tool(parse_docstring=True, error_on_invalid_docstring=False)`**：都会尝试解析 `Args:`，解析不了也不报错。四种写法对比如下：

| 写法 | 解析 `Args:` 吗 | docstring 不合规时 |
|---|---|---|
| 普通函数（不加装饰器） | 解析 | 整段 docstring 当描述，不报错 |
| `@tool` | 不解析 | 不涉及，整段 docstring 总是当描述 |
| `@tool(parse_docstring=True)` | 解析 | **报错** |
| `@tool(parse_docstring=True, error_on_invalid_docstring=False)` | 解析 | 整段 docstring 当描述，不报错 |

建议使用 `parse_docstring=True` 并保留默认的 `error_on_invalid_docstring=True`。这样格式问题在定义工具时就会暴露出来，不会悄悄把参数说明弄丢。

### 3.2.3 name_or_callable：更改工具名称

默认情况，使用函数名作为工具名称，但可以向@tool 传参 name_or_callable ，以更改工具名称。

In [12]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
@tool(parse_docstring=True,name_or_callable="getWeather")
def get_weather(city: str, units: str = "celsius", include_forecast: bool =
False) -> str:
    """
    获取当日天气，可选择是否同时查询未来五日天气预报
    
    Args:
        city: 城市
        units: 气温单位，可选：celsius-摄氏度，fahrenheit-华氏度
        include_forecast: 是否包含未来五日的天气预报
    """
    temp = 22 if units == "celsius" else 72
    result = f'{city}当天气温: {temp} {"摄氏度" if units == "celsius" else "华氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result
rprint(convert_to_openai_tool(get_weather))  


{
    'type': 'function',
    'function': {
        'name': 'getWeather',
        'description': '获取当日天气，可选择是否同时查询未来五日天气预报',
        'parameters': {
            'properties': {
                'city': {'description': '城市', 'type': 'string'},
                'units': {
                    'default': 'celsius',
                    'description': '气温单位，可选：celsius-摄氏度，fahrenheit-华氏度',
                    'type': 'string'
                },
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五日的天气预报',
                    'type': 'boolean'
                }
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

说明：@tool中参数name_or_callable名称可以省略，直接把名字作为第一个位置参数：

In [13]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
@tool("getWeather",parse_docstring=True)
def get_weather(city: str, units: str = "celsius", include_forecast: bool =
False) -> str:
    """
    获取当日天气，可选择是否同时查询未来五日天气预报
    
    Args:
        city: 城市
        units: 气温单位，可选：celsius-摄氏度，fahrenheit-华氏度
        include_forecast: 是否包含未来五日的天气预报
    """
    temp = 22 if units == "celsius" else 72
    result = f'{city}当天气温: {temp} {"摄氏度" if units == "celsius" else "华氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result
rprint(convert_to_openai_tool(get_weather))  


{
    'type': 'function',
    'function': {
        'name': 'getWeather',
        'description': '获取当日天气，可选择是否同时查询未来五日天气预报',
        'parameters': {
            'properties': {
                'city': {'description': '城市', 'type': 'string'},
                'units': {
                    'default': 'celsius',
                    'description': '气温单位，可选：celsius-摄氏度，fahrenheit-华氏度',
                    'type': 'string'
                },
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五日的天气预报',
                    'type': 'boolean'
                }
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

为什么这个参数叫 `name_or_callable`？因为 `@tool` 不带括号时，第一个参数收到的是函数本身（callable）；带括号写名字时，收到的是字符串（name）。详见 3.1.1。

起名建议：

- 模型要靠工具名和描述来选择工具，名字要见名知意
- 只用英文字母、数字、下划线和短横线。OpenAI 的接口文档要求函数名符合 `^[a-zA-Z0-9_-]{1,64}$`。LangChain 本身不检查，写中文名在定义时不会报错，但请求模型时可能被接口拒绝
- 同一个模型绑定的多个工具，名字要互不相同：执行工具时要靠名字找到对应的工具

### 3.2.4 args_schema：用 Pydantic 模型定义参数

默认情况下，参数说明是根据函数签名和 docstring 推断出来的，能表达的只有类型、默认值和一句说明。如果想告诉模型"`units` 只能取这两个值""`days` 必须在 1~7 之间"，就要自己写一个 Pydantic 模型，通过 `args_schema` 传进去。

这样做有两个好处：

1. **说明更精确**：`Literal` 会变成 `enum`，`ge` / `le` 会变成 `minimum` / `maximum`，模型能看到这些约束
2. **校验更严格**：模型传错参数时，`invoke` 在调用函数之前就会报错

In [14]:
from typing import Literal
from pydantic import BaseModel, Field

class WeatherInput(BaseModel):
    """查询天气的参数"""
    city: str = Field(description="城市名称，如：北京、上海")
    units: Literal["celsius", "fahrenheit"] = Field(default="celsius", description="气温单位")
    days: int = Field(default=1, ge=1, le=7, description="查询未来几天，范围 1~7")

@tool(args_schema=WeatherInput)
def get_weather(city: str, units: str = "celsius", days: int = 1) -> str:
    """获取指定城市的天气预报"""
    return f"{city}未来{days}天都是晴天，单位：{units}"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取指定城市的天气预报',
        'parameters': {
            'properties': {
                'city': {'description': '城市名称，如：北京、上海', 'type': 'string'},
                'units': {
                    'default': 'celsius',
                    'description': '气温单位',
                    'enum': ['celsius', 'fahrenheit'],
                    'type': 'string'
                },
                'days': {
                    'default': 1,
                    'description': '查询未来几天，范围 1~7',
                    'maximum': 7,
                    'minimum': 1,
                    'type': 'integer'
                }
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

参数合规时正常执行；违反约束时，函数还没执行就报错了：

In [15]:
print(get_weather.invoke({"city": "北京", "days": 3}))

for bad_args in [{"city": "北京", "days": 10}, {"city": "北京", "units": "kelvin"}]:
    try:
        get_weather.invoke(bad_args)
    except ValidationError as e:
        print(bad_args, "->", " ".join(line.strip() for line in str(e).splitlines()[1:3]))

北京未来3天都是晴天，单位：celsius
{'city': '北京', 'days': 10} -> days Input should be less than or equal to 7 [type=less_than_equal, input_value=10, input_type=int]
{'city': '北京', 'units': 'kelvin'} -> units Input should be 'celsius' or 'fahrenheit' [type=literal_error, input_value='kelvin', input_type=str]


使用 `args_schema` 的注意事项：

- **字段名必须和函数参数名一致。** 调用时，校验后的字段会按名字传给函数。LangChain 不检查两者是否对得上，定义时不会报错，要等到调用时才出现 `TypeError: got an unexpected keyword argument`
- 参数说明写在 `Field(description=...)` 里，就不需要 `parse_docstring` 了
- 函数没有 docstring、又没传 `description` 时，会改用 **`args_schema` 类的 docstring** 作为工具描述（这里就是"查询天气的参数"），这通常不是你想要的。所以函数的 docstring 最好照常写

### 3.2.5 return_direct：直接返回工具结果

`return_direct` 只在 **Agent 循环**中起作用。回顾第 1 节的工具调用流程：正常情况下，工具执行完要把结果交回模型，由模型组织最终回答。设置 `return_direct=True` 后，工具执行完就直接结束，工具的返回值就是最终答案：

```text
默认：              用户提问 → 模型发起 tool_call → 执行工具 → 结果交回模型 → 模型生成最终回答
return_direct=True：用户提问 → 模型发起 tool_call → 执行工具 → 结束（工具结果就是最终答案）
```

适用场景：工具的结果已经可以直接给用户看，不需要模型再加工，比如查询订单状态、生成固定格式的报告。好处是少调用一次模型，更快、更省 token，结果也不会被模型改写。

注意：`return_direct` 只是存在工具对象上的一个标记，**不会发给模型**（`convert_to_openai_tool` 的输出里没有它），由 Agent 框架在执行完工具后读取。如果像 `01-Tools_usage.ipynb` 那样手动执行工具，除非自己的代码去检查它，否则不会有任何效果。

下面用 `create_agent` 对比一下。Agent 会在后面的章节细讲，这里只看效果。模型用 DeepSeek（配置与 `01-Tools_usage.ipynb` 相同）：

In [ ]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langchain.agents import create_agent

load_dotenv(override=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

ds_model = ChatDeepSeek(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    model_name="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}},
)

def query_weather(city: str) -> str:
    """查询指定城市的天气"""
    return f"{city}：晴，22 摄氏度，东南风 2 级"

# 同一个函数做成两个工具。tool(...)(函数) 就是 @tool(...) 的等价写法（见 3.1.1）
normal_tool = tool("get_weather")(query_weather)
direct_tool = tool("get_weather", return_direct=True)(query_weather)

for t in [normal_tool, direct_tool]:
    agent = create_agent(ds_model, tools=[t])
    result = agent.invoke({"messages": [{"role": "user", "content": "北京天气怎么样？"}]})
    print(f"===== return_direct={t.return_direct} =====")
    for m in result["messages"]:
        print(f"{type(m).__name__:<13}", m.tool_calls if getattr(m, "tool_calls", None) else m.content)

运行上面的代码，对比两组消息列表：

```text
===== return_direct=False =====
HumanMessage  北京天气怎么样？
AIMessage     [{'name': 'get_weather', 'args': {'city': '北京'}, ...}]   ← 第 1 次调用模型：发起工具调用
ToolMessage   北京：晴，22 摄氏度，东南风 2 级                            ← 工具结果交回模型
AIMessage     （模型组织的最终回答，每次运行措辞可能不同）                 ← 第 2 次调用模型

===== return_direct=True =====
HumanMessage  北京天气怎么样？
AIMessage     [{'name': 'get_weather', 'args': {'city': '北京'}, ...}]   ← 第 1 次调用模型：发起工具调用
ToolMessage   北京：晴，22 摄氏度，东南风 2 级                            ← 到此结束，这就是最终结果
```

- 默认情况下，消息列表以模型的最终回答（`AIMessage`）结尾，模型一共被调用 2 次
- `return_direct=True` 时，消息列表在 `ToolMessage` 处就结束了：最终结果是工具的原始返回值，模型只被调用 1 次

这个判断写在 `create_agent` 的源码里（`langchain/agents/factory.py`）：执行完工具后，如果这一轮执行的工具**全部**设置了 `return_direct=True`，就结束循环；否则回到模型。

### 3.2.6 response_format：区分"给模型的内容"和"附件"

默认情况下（`response_format="content"`），函数的返回值会全部作为 `ToolMessage.content` 交回模型。但有时工具拿到的原始数据很大，或者只有程序需要（比如完整的接口响应、检索到的原始文档），不想全部塞给模型。

设置 `response_format="content_and_artifact"` 后，函数要返回一个**二元组** `(content, artifact)`：

- `content`：给模型看的内容，放进 `ToolMessage.content`
- `artifact`：附件，放进 `ToolMessage.artifact`。它**不会发给模型**，留给程序后续使用

In [17]:
@tool(response_format="content_and_artifact")
def search_weather(city: str) -> tuple[str, dict]:
    """查询指定城市的天气"""
    raw = {"city": city, "temp": 22, "humidity": 0.45, "wind": "东南风 2 级", "source": "demo-api"}
    content = f"{city}今天晴，22 摄氏度"   # 只把模型需要的信息交给模型
    return content, raw                    # 返回 (content, artifact) 二元组

# 传入 tool_call：得到带附件的 ToolMessage
tool_call = {"name": "search_weather", "args": {"city": "北京"}, "id": "call_456", "type": "tool_call"}
msg = search_weather.invoke(tool_call)
print("content :", msg.content)
print("artifact:", msg.artifact)

# 传入普通字典：只返回 content，附件被丢弃
print("普通调用:", search_weather.invoke({"city": "北京"}))

content : 北京今天晴，22 摄氏度
artifact: {'city': '北京', 'temp': 22, 'humidity': 0.45, 'wind': '东南风 2 级', 'source': 'demo-api'}
普通调用: 北京今天晴，22 摄氏度


两点注意：

- **只有传入 tool_call 调用时，才能拿到附件。** 传普通字典调用时，结果不会包装成 `ToolMessage`，只返回 content。Agent 框架执行工具时传的都是 tool_call，所以不受影响
- 设置了 `content_and_artifact`，函数就必须返回二元组，否则调用时报 `ValueError`

## 3.3 总结

**原理：**

1. `@tool` 是 `get_weather = tool(get_weather)` 的简写。装饰器返回什么，`get_weather` 就变成什么：`tool()` 返回的是 `StructuredTool` 对象，原函数存在它的 `func` 属性里。
2. `tool()` 根据第一个参数判断用法：收到函数就直接创建工具（`@tool`）；收到名字或什么都没收到，就先返回一个装饰器（`@tool("名字")`、`@tool(参数=...)`）。
3. 工具说明在**定义函数时**就生成好了（`StructuredTool.from_function`）：函数名 → `name`，docstring → `description`，函数签名 → `args_schema`（Pydantic 模型）。
4. 工具对象要用 `.invoke()` 调用：先用 `args_schema` 校验参数，再调用原函数。传入 tool_call 时，直接返回可以回传给模型的 `ToolMessage`。

**常用参数速查：**

| 参数 | 一句话 | 例子 |
|---|---|---|
| `description` | 覆盖 docstring，作为工具描述 | `@tool(description="根据城市查询天气")` |
| `parse_docstring` | 把 `Args:` 解析成参数说明，docstring 必须合规 | `@tool(parse_docstring=True)` |
| `error_on_invalid_docstring` | docstring 不合规时是否报错 | `@tool(parse_docstring=True, error_on_invalid_docstring=False)` |
| `name_or_callable` | 自定义工具名，参数名可以省略 | `@tool("getWeather")` |
| `args_schema` | 用 Pydantic 模型精确定义参数和约束 | `@tool(args_schema=WeatherInput)` |
| `return_direct` | 在 Agent 中，工具执行完直接结束 | `@tool(return_direct=True)` |
| `response_format` | 返回（给模型的内容，附件） | `@tool(response_format="content_and_artifact")` |

**推荐写法：**

```python
@tool(parse_docstring=True)
def get_weather(city: str, units: str = "celsius") -> str:
    """获取指定城市的当日天气。

    Args:
        city: 城市名称，如：北京、上海
        units: 气温单位，可选 celsius 或 fahrenheit
    """
    ...
```

参数需要枚举、取值范围等约束时，改用 `args_schema`。普通函数和 `@tool` 的完整对比，见 `02-tool_define.ipynb` 的 2.4.1 节。